# Experiment Audit — Sentiment140

Reconstrucción del protocolo, runs, contribuciones y modelo final desde MLflow.


In [ ]:
import os
import math
import pandas as pd
import mlflow
from mlflow import MlflowClient

MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "http://ec2-100-26-91-142.compute-1.amazonaws.com:5000"
)

EXPERIMENT_NAME = "nlp-lab2-sentiment140"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_TRACKING_URI)

client = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI,
    registry_uri=MLFLOW_TRACKING_URI
)

print("Tracking URI:", mlflow.get_tracking_uri())


## Protocolo


In [ ]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None

protocol_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.lab_run_type = 'protocol'"
)

assert len(protocol_runs) == 1

protocol = protocol_runs[0]

print("Protocol run:", protocol.info.run_id)
print(protocol.data.params)


## Runs presentados


In [ ]:
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id]
)

presented = [
    r for r in runs
    if r.data.tags.get("lab_run_type")
    in {"protocol", "experiment", "final"}
]

rows = []

for r in presented:
    rows.append({
        "run_id": r.info.run_id,
        "status": r.info.status,
        "run_type": r.data.tags.get("lab_run_type"),
        "experiment": r.data.tags.get("lab_experiment_id"),
        "stage": r.data.tags.get("lab_stage"),
        "member": r.data.tags.get("lab_member_id"),
        "configuration_id": r.data.tags.get("lab_configuration_id"),
        "macro_f1_mean": r.data.metrics.get("macro_f1_mean"),
        "test_macro_f1": r.data.metrics.get("test_macro_f1"),
    })

display(
    pd.DataFrame(rows)
    .sort_values("run_id")
)


## Modelo final


In [ ]:
champion = client.get_model_version_by_alias(
    "sentiment140",
    "champion"
)

final_run = client.get_run(
    champion.run_id
)

print("Model:", champion.name)
print("Version:", champion.version)
print("Run:", champion.run_id)
print(
    "Training size:",
    final_run.data.params.get("training_size")
)
print(
    "Test Macro-F1:",
    final_run.data.metrics.get("test_macro_f1")
)
